# U.S. Secretary of State Speech Corpus: quick start (Python)

This notebook loads the corpus and walks through a first look at it: who speaks, when, where, how much, about which countries and people, and how to follow a word over time. Every section runs in seconds on a laptop.

**Before you start**

```bash
pip install sos-speeches pandas matplotlib
```

**How a row is organised.** One row is one public statement by a Secretary of State: a speech, remarks, an interview, testimony, a press statement or an op-ed. The two text columns differ in who is speaking:

- `SOS Text` holds only the Secretary's own words. Use it to study what the Secretary said.
- `Natural Text` holds the full transcript, every speaker included.

The other columns hold the document's metadata (`Secretary`, `Date`, `Title`, `Speech Type`, `Location`, `URL`) and fields derived from the text: word counts, `Speakers Present`, and the people and countries mentioned. The full codebook is a PDF attached to each [GitHub release](https://github.com/revolins/sos-corpus/releases).

## 1. Load the data

In [ ]:
import csv
import os
import re

import matplotlib.pyplot as plt
import pandas as pd

csv.field_size_limit(10 ** 8)          # some text fields are very long
pd.set_option('display.max_colwidth', 120)

# Either point SOS_CORPUS_CSV at a local copy of the CSV, or let the loader
# download the latest GitHub release once (it is cached for later runs).
path = os.environ.get('SOS_CORPUS_CSV')
if not path:
    import sos_speeches
    path = sos_speeches.load()

# Raw Text (the full scraped page) is large and rarely needed for analysis,
# so it is left out here. Drop the usecols argument to keep it.
df = pd.read_csv(path, dtype=str, keep_default_na=False,
                 usecols=lambda c: c != 'Raw Text')
for c in ['SOS Word Count', 'Natural Word Count']:
    df[c] = df[c].astype(int)
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
print(f'{len(df):,} rows, {df.shape[1]} columns, {df.Date.min():%Y-%m-%d} to {df.Date.max():%Y-%m-%d}')
print(os.path.basename(path))

In [ ]:
# Secretaries in order of tenure. The corpus also keeps one row for Warren
# Christopher (his farewell testimony, 1997-01-08); it is left out of the plots.
ORDER = ['Albright', 'Powell', 'Rice', 'Clinton', 'Kerry', 'Tillerson', 'Pompeo', 'Blinken', 'Rubio']
d = df[df.Secretary.isin(ORDER)].copy()
d['Secretary'] = pd.Categorical(d.Secretary, ORDER, ordered=True)
d.dtypes

In [ ]:
# One row, in full
row = d.sample(1, random_state=7).iloc[0]
print(row[['Secretary', 'Date', 'Title', 'Speech Type', 'Location', 'URL']].to_string())
print('\nSpeakers Present:', row['Speakers Present'])
print('\nSOS Text (first 600 characters):\n', row['SOS Text'][:600])

## 2. Who and when

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
d.Secretary.value_counts(sort=False).plot.bar(ax=ax[0], color='#3b6ea5')
ax[0].set_title('Statements per Secretary'); ax[0].set_ylabel('rows')
d.groupby('Year').size().plot.bar(ax=ax[1], color='#3b6ea5', width=0.85)
ax[1].set_title('Statements per year'); ax[1].set_xlabel('')
ax[1].tick_params(axis='x', labelsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# What kind of statement each Secretary gives (share of their rows, %)
(pd.crosstab(d.Secretary, d['Speech Type'], normalize='index') * 100).round(1)

## 3. How much the Secretary says

`SOS Word Count` counts the Secretary's own words. `Natural Word Count` counts the whole transcript. Their ratio shows how much of an event is the Secretary talking: close to 1 for a speech or a written statement, much lower for a press conference.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
d.boxplot(column='SOS Word Count', by='Secretary', ax=ax[0], showfliers=False)
ax[0].set_title("Secretary's own words per statement"); ax[0].set_xlabel(''); ax[0].set_yscale('log')
share = (d['SOS Word Count'] / d['Natural Word Count'].clip(lower=1)).clip(upper=1)
share.groupby(d['Speech Type']).median().sort_values().plot.barh(ax=ax[1], color='#3b6ea5')
ax[1].set_title("Median share of the transcript spoken by the Secretary")
plt.suptitle(''); plt.tight_layout(); plt.show()
d.groupby('Secretary', observed=True)['SOS Word Count'].agg(['count', 'sum', 'median'])

## 4. Where

`Location` is always `City, Region`. Written statements issued by the department carry `Washington, D.C.`.

In [ ]:
print(d.Location.value_counts().head(15).to_string())
abroad = (d.Location != 'Washington, D.C.').groupby(d.Secretary, observed=True).mean().mul(100).round(1)
abroad.rename('% of statements given outside Washington')

## 5. Which countries and people come up

Four columns hold lists, joined by `"; "`: `Speakers Present`, `People Mentioned (Natural Text)`, `People Mentioned (SOS Text)` and `Countries Mentioned`. `Country ISO Codes Mentioned` holds the matching ISO codes in the same order.

The helper below turns any of these lists into one row per entry.

`People Mentioned` comes from automatic named-entity recognition. Its frequent values are reviewed, but treat the long tail as approximate.

In [ ]:
def explode(frame, column):
    """One row per entry of a '; '-joined list column."""
    out = frame[['Row Id', 'Secretary', 'Year', column]].copy()
    out[column] = out[column].str.split('; ')
    out = out.explode(column)
    return out[out[column].astype(bool)]

countries = explode(d, 'Countries Mentioned')
print(countries['Countries Mentioned'].value_counts().head(15).to_string())

In [ ]:
# The five countries each Secretary mentions in the most statements
# (the United States itself left out)
c = countries[countries['Countries Mentioned'] != 'United States']
top = (c.groupby('Secretary', observed=True)['Countries Mentioned']
         .apply(lambda s: ', '.join(s.value_counts().head(5).index)))
top.rename('most-mentioned countries')

In [ ]:
people = explode(d, 'People Mentioned (SOS Text)')
people['People Mentioned (SOS Text)'].value_counts().head(20)

## 6. Following a word over time

The plot counts how often each term appears per 10,000 of the Secretary's own words, by year. Normalising by words matters because the number of statements changes a lot from year to year. Edit `TERMS` to follow anything.

In [ ]:
TERMS = {'China': r'\bchin(?:a|ese)\b', 'Russia': r'\bruss(?:ia|ian)\b',
         'Iran': r'\biran(?:ian)?\b', 'climate': r'\bclimate\b'}
text = d['SOS Text'].str.lower()
words_per_year = d.groupby('Year')['SOS Word Count'].sum()
rates = pd.DataFrame({name: text.str.count(rx).groupby(d.Year).sum() / words_per_year * 1e4
                      for name, rx in TERMS.items()})
ax = rates.plot(figsize=(11, 4), marker='.')
ax.set_ylabel('mentions per 10,000 words'); ax.set_title("Terms in the Secretary's own words")
plt.show()

In [ ]:
def kwic(term, secretary=None, width=70, n=8):
    """Keyword in context: each match shown with the words around it."""
    rows = d if secretary is None else d[d.Secretary == secretary]
    rx = re.compile(term, re.I)
    shown = 0
    for _, r in rows.sort_values('Date').iterrows():
        for m in rx.finditer(r['SOS Text']):
            s = r['SOS Text']
            left = s[max(0, m.start() - width):m.start()].replace('\n', ' ')
            right = s[m.end():m.end() + width].replace('\n', ' ')
            print(f"{r['Date']:%Y-%m-%d} {r['Secretary']:<9} …{left}[{m.group(0)}]{right}…")
            shown += 1
            if shown >= n:
                return

kwic(r'\bArctic\b', n=6)

## 7. Before you analyse

- **Cite rows by `Row Hash`.** It changes only when the content changes. `Row Id` is renumbered when rows are added or removed between releases.
- **Use `URL` for provenance** and `Canonical URL` to open the page today; state.gov has moved some paths.
- **Text cleanup.** `SOS Text` has speaker labels and stage directions such as "(Applause.)" removed. Interviewers' questions are in `Natural Text` only.
- **`Date` is the publication date.** A statement released the evening after an event may be dated a day later.
- **Updates.** New Rubio statements are added every Monday, and each update is a new versioned release. Note which release you used, e.g. `corpus_v2026.10.27.csv`.
- **Releases and verification.** See the [README](https://github.com/revolins/sos-corpus#readme) for the release notes and the verification checks every release passes.